##### Task 13
Build a gold orders table.

What I want in the end:
- One row per order
- Fully enriched with customer data
- Includes aggregated order metrics
- Only valid, trusted records
- Saved as something like `gold_orders`

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
-- Temp view for order metrics: total_items, total_value, average_item_value, distinct_items

CREATE OR REPLACE TEMP VIEW order_metrics AS
SELECT 
  o.order_id, 
  SUM(o.quantity) total_items,
  SUM(o.quantity * i.price) total_value,
  ROUND(total_value / total_items, 2) avg_item_value,
  COUNT(DISTINCT(i.item_id)) distinct_items
FROM silver_order_lines o
LEFT JOIN silver_items i
  ON o.item_id = i.item_id
GROUP BY o.order_id;

SELECT *
FROM order_metrics;

In [0]:
%sql
SELECT *
FROM silver_order_lines;

In [0]:
%sql
-- Build a gold orders table

CREATE OR REPLACE TABLE gold_orders AS
SELECT o.order_id, o.order_date, o.customer_id, o.status, o.items,
  c.name, c.email, c.country,
  m.total_items, m.total_value, m.avg_item_value, m.distinct_items
FROM silver_orders o
LEFT JOIN silver_customers c
  ON o.customer_id = c.customer_id
LEFT JOIN order_metrics m
  ON o.order_id = m.order_id;

SELECT *
FROM gold_orders;